# Separability, Gaussian error probabilities, and the complete figure

Run after notebook 01_plot. This notebook independently reads every saved
waveform sample and computes complete nonoverlapping integration-window means
and sample variances (ddof=1). It calculates pooled-variance separability d′,
midpoint thresholds, class-conditional Gaussian PFA/PFN and average decision
error at all 94 integration widths for each of three signal/baseline pairs.

The solid/dashed 1/N support convention and all original visual settings are
preserved. Gaussian tail probabilities are model estimates, not empirical
measurements. The analysis uses the original 4-million-sample block order.

It combines these freshly computed results with notebook 01's `study_density.npz`
to produce `capacitance_detection_noise_1p5_irregular_drift.pdf` and
PNG, plus detection CSV/JSON, per-source statistics, and `study.json`.
The cached inset is checked against samples read directly from the waveforms.
The density archive must be produced by `01_plot` for the same waveforms and
plotting settings; its shape is checked, but full density bins are not recomputed here.

Dependencies: Python 3.10+, NumPy, SciPy, Numba, Matplotlib and PyMuPDF.
All analysis and figure assembly code is included; no repository modules or
code from another notebook are imported. Rerunning replaces derived outputs.

The verified environment uses Python 3.12.13, NumPy 2.5.3, SciPy 1.18.1,
Numba 0.67.0, Matplotlib 3.11.1 and PyMuPDF 1.28.2, with local timezone
America/Detroit (UTC−04:00). Package and timezone differences can change
PDF serialization; `execution_environment.json` records the actual run.


Each source is streamed once to measure its integer sum, sum of squares,
sample count, center and inset, then again for integration-window statistics.
The second pass independently verifies those moments. No saved statistics or
external JSON files are read. Each integration width uses its measured signal
and baseline means for separation and midpoint thresholds. Centering is only a
numerical technique for stable variance calculations; it does not modify samples.


In [ ]:
from __future__ import annotations
import hashlib
import io
import json
import math
import os
from pathlib import Path
import tempfile
import time
import zipfile
from dataclasses import dataclass
import numpy as np

import csv
from numba import njit
from scipy.special import ndtr, log_ndtr
VERSION = '13.0'
MIN_FONT_SIZE = 18.0
DEFAULT_BRIGHTNESS_GAMMA = DEFAULT_BRIGHTNESS_EXPOSURE = 1.0
BRIGHTNESS_REFERENCE_PERCENTILE = 99.9
VISIBLE_SOURCES = (0, 1, 2, 3)
SUPPORT_MODES = ('sample_count', 'independent_windows')



In [ ]:
INPUT = Path('waveform_parts')  # Or fresh_run3/complete_waveforms.npz
OUTPUT_DIR = Path('./')
ANALYSIS_CHUNK = 4_000_000
VERIFY_CHECKSUMS = True



# All settings needed for the figure; no generation metadata is read.
SETTINGS = {'sample_rate_hz': 10000.0,
 'points_per_recording': [1000000000, 1000000000, 1000000000, 1000000000],
 'main_view': {'start': 0,
               'stop': 1000000000,
               'nx': 1600,
               'ny': 1000,
               'ymin': -35.0,
               'ymax': 5.0},
 'inset_view': {'start': 540000000, 'stop': 540000400, 'ymin': -15.0, 'ymax': 5.0},
 'main_y_ticks_codes': [-35, -30, -25, -20, -15, -10, -5, 0, 5],
 'main_panel_sources': [0, 1, 2, 3],
 'delta_c_ff': [0.0, 50.0, 75.0, 100.0],
 'waveform_threshold_codes': -6.35,
 'render_settings': {'gamma': 0.8, 'exposure': 1.0},
 'analysis': {'support_mode': 'sample_count'}}


Dataset reader and original integration-window and Gaussian probability kernels.


In [3]:

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024**2), b''):
            digest.update(block)
    return digest.hexdigest()


class WaveformInput:
    """Discover self-describing NPZ parts; no manifest or generation report is needed."""
    def __init__(self, path, verify_checksums=True, settings=None):
        self.path = Path(path).resolve()
        self.verify_checksums = verify_checksums
        self.metadata = json.loads(json.dumps(SETTINGS if settings is None else settings))
        self.parts = None
        counts = self.metadata['points_per_recording']
        if len(counts) != 4:
            raise ValueError('Four recordings are required.')
        if self.path.is_dir():
            self.parts = []
            for path in sorted(self.path.glob('*.npz')):
                with np.load(path, allow_pickle=False) as archive:
                    sid_array, start_array = archive['source_index'], archive['start_sample']
                    if (sid_array.shape != () or start_array.shape != ()
                            or sid_array.dtype.kind not in 'iu' or start_array.dtype.kind not in 'iu'):
                        raise ValueError(f'Source ID and sample offset must be integer scalars: {path}')
                    sid, start = int(sid_array.item()), int(start_array.item())
                with zipfile.ZipFile(path) as archive:
                    with archive.open('waveform.npy') as handle:
                        shape, order, dtype = self.array_header(handle)
                if (sid not in range(4) or start < 0 or len(shape) != 1 or shape[0] < 1
                        or order or dtype != np.dtype('<i2')):
                    raise ValueError(f'Invalid waveform part: {path}')
                self.parts.append(dict(file=path.name, source_index=sid, start=start,
                    count=shape[0], bytes=path.stat().st_size, sha256=sha256_file(path)))
            self.parts.sort(key=lambda p: (p['source_index'], p['start']))
            for sid, count in enumerate(counts):
                expected = 0
                for part in (p for p in self.parts if p['source_index'] == sid):
                    if part['start'] != expected:
                        raise ValueError(f'Source {sid}: missing, duplicate or overlapping part at sample {expected:,}.')
                    expected += part['count']
                if expected != count:
                    raise ValueError(f'Source {sid}: found {expected:,} samples; expected {count:,}.')
            # Hash source order, ranges and actual file contents, independently
            # of filenames. Notebook 03 compares this with notebook 02's output.
            identity = [{key: part[key] for key in ('source_index','start','count','sha256')}
                        for part in self.parts]
            self.fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
        elif self.path.is_file() and self.path.suffix.lower() == '.npz':
            with zipfile.ZipFile(self.path) as archive:
                for sid, count in enumerate(counts):
                    with archive.open(f'source_{sid}.npy') as handle:
                        shape, order, dtype = self.array_header(handle)
                    if shape != (count,) or order or dtype != np.dtype('<i2'):
                        raise ValueError(f'Invalid source {sid} array header.')
            # Any legacy metadata member is deliberately ignored.
            self.fingerprint = sha256_file(self.path)
        else:
            raise ValueError('INPUT must be a directory of NPZ parts or a single waveform NPZ file.')

    @staticmethod
    def array_header(handle):
        version = np.lib.format.read_magic(handle)
        if version == (1, 0):
            return np.lib.format.read_array_header_1_0(handle)
        if version == (2, 0):
            return np.lib.format.read_array_header_2_0(handle)
        raise ValueError(f'Unsupported NPY header version: {version}')

    @staticmethod
    def read_exact(handle, count):
        data = bytearray(count)
        view = memoryview(data)
        position = 0
        while position < count:
            amount = handle.readinto(view[position:])
            if not amount:
                raise EOFError('Truncated waveform array.')
            position += amount
        return data

    def pieces(self, sid, chunk):
        if self.parts is not None:
            for part in (p for p in self.parts if p['source_index'] == sid):
                path = self.path / part['file']
                if self.verify_checksums and sha256_file(path) != part['sha256']:
                    raise ValueError(f'Waveform part changed while processing: {path}')
                with np.load(path, allow_pickle=False) as archive:
                    values = archive['waveform']
                    if (values.shape != (part['count'],) or values.dtype != np.dtype('<i2')
                            or archive['source_index'].item() != sid
                            or archive['start_sample'].item() != part['start']):
                        raise ValueError(f'Incorrect waveform header: {path}')
                yield values
        else:
            with zipfile.ZipFile(self.path) as archive:
                with archive.open(f'source_{sid}.npy') as handle:
                    shape, order, dtype = self.array_header(handle)
                    count = self.metadata['points_per_recording'][sid]
                    if shape != (count,) or order or dtype != np.dtype('<i2'):
                        raise ValueError(f'Incorrect source {sid} array header.')
                    for start in range(0, count, chunk):
                        size = min(chunk, count-start)
                        yield np.frombuffer(self.read_exact(handle, 2*size), dtype='<i2')
                    if handle.read(1):
                        raise ValueError('Unexpected trailing waveform bytes.')

    def blocks(self, sid, chunk):
        """Reblock any part boundaries to the original analysis chunk size."""
        if chunk < 1:
            raise ValueError('Chunk size must be positive.')
        buffer = np.empty(chunk, dtype='<i2')
        filled = 0
        for piece in self.pieces(sid, chunk):
            offset = 0
            while offset < piece.size:
                amount = min(chunk-filled, piece.size-offset)
                buffer[filled:filled+amount] = piece[offset:offset+amount]
                filled += amount
                offset += amount
                if filled == chunk:
                    yield buffer
                    buffer = np.empty(chunk, dtype='<i2')
                    filled = 0
        if filled:
            yield buffer[:filled]


In [4]:
@dataclass(frozen=True)
class View:
    """Time interval [start,stop); amplitude limits are inclusive."""
    start: int
    stop: int
    nx: int
    ny: int
    ymin: float
    ymax: float

    def validate(self) -> None:
        if not (0 <= self.start < self.stop <= np.iinfo(np.int64).max):
            raise ValueError("A view needs 0 <= start < stop within int64 range.")
        if not (1 <= self.nx <= 65536 and 1 <= self.ny <= 65536):
            raise ValueError("Each histogram dimension must be in [1,65536].")
        if (self.stop - self.start) * self.nx > np.iinfo(np.int64).max:
            raise ValueError("Time span times horizontal bins exceeds int64.")
        _validate_amplitude(self.ymin, self.ymax)
        if not math.isfinite(self.ny / (self.ymax - self.ymin)):
            raise ValueError("Amplitude scale is not representable.")

    @property
    def cells(self) -> int:
        return self.nx * self.ny

    def kernel_args(self) -> tuple:
        return (np.int64(self.start), np.int64(self.stop), np.int32(self.nx),
                np.int32(self.ny), np.float64(self.ymin), np.float64(self.ymax))


@dataclass(frozen=True)
class Inset:
    start: int
    stop: int
    ymin: float
    ymax: float

    @property
    def count(self) -> int:
        return self.stop - self.start

    def validate(self) -> None:
        if not (0 <= self.start < self.stop <= np.iinfo(np.int64).max):
            raise ValueError("Inset sample interval must be nonempty and within int64.")
        if self.count > 1_000_000:
            raise ValueError("Line inset is limited to 1,000,000 samples; default is 200.")
        _validate_amplitude(self.ymin, self.ymax)


def _validate_amplitude(lo: float, hi: float) -> None:
    if not (math.isfinite(lo) and math.isfinite(hi) and lo < hi
            and math.isfinite(hi - lo)):
        raise ValueError("Amplitude limits must be finite and increasing.")
@njit(cache=False, nogil=True)
def _update_windows(prefix, start, before, widths, last_prefix, counts, means,
                    m2s, center, gain, bias):
    """Consume a chunk using one int64 prefix sum for all integration widths.

    Previous complete-window prefix values cross I/O boundaries. Centered,
    two-pass block moments merge using the parallel variance identity, avoiding
    variance calculation by subtraction of large DC second moments.
    """
    stop = start + prefix.size
    for j in range(widths.size):
        width = widths[j]
        first = (start // width + 1) * width
        count = stop // width - start // width
        if count == 0:
            continue
        previous = last_prefix[j]
        total = 0.0
        for end in range(first, stop + 1, width):
            current = before + prefix[end - start - 1]
            deviation = ((current - previous) / width) * gain + bias - center
            total += deviation
            previous = current
        block_mean = total / count
        previous = last_prefix[j]
        block_m2 = 0.0
        for end in range(first, stop + 1, width):
            current = before + prefix[end - start - 1]
            deviation = ((current - previous) / width) * gain + bias - center
            difference = deviation - block_mean
            block_m2 += difference * difference
            previous = current
        last_prefix[j] = previous
        old_count = counts[j]
        new_count = old_count + count
        difference = block_mean - means[j]
        m2s[j] += block_m2 + difference * difference * (old_count * (count / new_count))
        means[j] += difference * (count / new_count)
        counts[j] = new_count


def window_sizes(n, fs, max_tau=1000.0, points=100, min_windows=10):
    """Unique rounded geometric widths, following notebook cell 1.

    The default duration/10 ceiling ensures at least ten complete windows.
    max_tau additionally limits the displayed domain while retaining all of the
    shorter recordings' valid widths through that domain.
    """
    if (not isinstance(n, (int, np.integer)) or n < 1
            or not math.isfinite(fs) or fs <= 0
            or not math.isfinite(max_tau) or max_tau <= 0
            or not isinstance(points, (int, np.integer)) or points < 1
            or not isinstance(min_windows, (int, np.integer)) or min_windows < 2):
        raise ValueError('Invalid integration arguments.')
    maximum = min(max_tau * fs, n // min_windows)
    if maximum < 1:
        raise ValueError('Insufficient samples for the requested integration windows.')
    widths = np.unique(np.round(np.geomspace(1, maximum, points)).astype(np.int64))
    return widths[(widths >= 1) & (n // widths >= min_windows)]


def _normal_tail(argument, sigma, deterministic_event):
    """Stable CDF and log-CDF, including a point mass when sigma is zero."""
    if sigma == 0:
        probability = float(bool(deterministic_event))
        return probability, (0.0 if probability else -math.inf)
    standardized = argument / sigma
    return float(ndtr(standardized)), float(log_ndtr(standardized))


def refresh_rows(rows, support_mode='sample_count'):
    """Return rows with d', midpoint threshold and Gaussian tails recalculated.

    Required measured inputs are mean_baseline, mean_signal, sigma_baseline,
    sigma_signal, pair_samples, baseline_windows and signal_windows. Cached d',
    thresholds and Gaussian probabilities are ignored. Variances are unbiased
    sample variances of the complete nonoverlapping window means.
    """
    if support_mode not in SUPPORT_MODES:
        raise ValueError(f'support_mode must be one of {SUPPORT_MODES}.')
    refreshed = []
    empirical_fields = (
        'false_positive_rate', 'false_negative_rate', 'decision_error',
        'false_positives', 'false_negatives', 'baseline_ties', 'signal_ties',
        'one_error_resolution', 'one_error_resolution_baseline',
        'one_error_resolution_signal', 'tie_policy')
    for original in rows:
        row = dict(original)
        mu0, mu1 = float(row['mean_baseline']), float(row['mean_signal'])
        sd0, sd1 = float(row['sigma_baseline']), float(row['sigma_signal'])
        if not all(math.isfinite(v) for v in (mu0, mu1, sd0, sd1)) or min(sd0, sd1) < 0:
            raise ValueError('Measured means and standard deviations must be finite; sigmas nonnegative.')
        pair_samples = int(row['pair_samples'])
        blocks0, blocks1 = int(row['baseline_windows']), int(row['signal_windows'])
        if pair_samples < 1 or min(blocks0, blocks1) < 2 or blocks0 != blocks1:
            raise ValueError('Pair-matched recordings require equal counts of at least two windows.')
        pooled_sd = math.hypot(sd0, sd1) / math.sqrt(2)
        separation = abs(mu1 - mu0)
        dprime = separation / pooled_sd if pooled_sd else (math.inf if separation else 0.0)
        threshold = 0.5 * mu0 + 0.5 * mu1
        direction = 1 if mu1 >= mu0 else -1
        baseline_margin = direction * (mu0 - threshold)
        signal_margin = direction * (mu1 - threshold)
        pfa, log_pfa = _normal_tail(baseline_margin, sd0, baseline_margin > 0)
        pfn, log_pfn = _normal_tail(-signal_margin, sd1, signal_margin <= 0)
        effective_trials = pair_samples if support_mode == 'sample_count' else blocks0
        for field in empirical_fields:
            row.pop(field, None)
        row.update(
            pair_samples=pair_samples, baseline_windows=blocks0, signal_windows=blocks1,
            mean_baseline=mu0, mean_signal=mu1, sigma_baseline=sd0, sigma_signal=sd1,
            variance_baseline=sd0 * sd0, variance_signal=sd1 * sd1,
            pooled_sigma=pooled_sd, measured_separation_codes=separation,
            dprime=dprime, threshold_codes=threshold,
            threshold_policy='pair_midpoint_per_tau',
            detection_direction='upper' if direction == 1 else 'lower',
            decision_tie_policy='baseline',
            gaussian_false_positive_rate=pfa,
            gaussian_false_negative_rate=pfn,
            gaussian_decision_error=0.5 * pfa + 0.5 * pfn,
            log_gaussian_false_positive_rate=log_pfa,
            log_gaussian_false_negative_rate=log_pfn,
            pfa_limit=1.0 / effective_trials, effective_trials=effective_trials,
            support_mode=support_mode, gaussian_model='class_conditional_midpoint',
            variance_ddof=1)
        refreshed.append(row)
    return refreshed


In [5]:
def measure_source(reader, sid, chunk):
    """Measure full-record integer moments and the inset directly from samples."""
    expected = reader.metadata['points_per_recording'][sid]
    inset = Inset(**reader.metadata['inset_view'])
    inset.validate()
    if inset.stop > expected:
        raise ValueError('Inset extends beyond the recording.')
    trace = np.empty(inset.count, dtype=np.int16)
    total = squared = consumed = captured = 0
    for values in reader.blocks(sid, chunk):
        integers = values.astype(np.int64)
        total += int(integers.sum(dtype=np.int64))
        squared += int(np.square(integers).sum(dtype=np.int64))
        lo, hi = max(consumed, inset.start), min(consumed + values.size, inset.stop)
        if lo < hi:
            trace[lo-inset.start:hi-inset.start] = values[lo-consumed:hi-consumed]
            captured += hi-lo
        consumed += values.size
    if consumed != expected or captured != inset.count:
        raise ValueError(f'Source {sid}: incomplete recording or inset.')
    print(f'Source {sid}: measured {consumed:,} samples directly from the waveform.', flush=True)
    return dict(samples=consumed, integer_sum=total, integer_sum_of_squares=squared), trace


def analyze_source(reader, sid, chunk):
    metadata = reader.metadata
    count = metadata['points_per_recording'][sid]
    fs = metadata['sample_rate_hz']
    check = metadata['source_checks'][sid]
    # Measured full-record mean is a numerical offset for stable moments only.
    # It is added back below; no samples or waveform amplitudes are changed.
    center = check['integer_sum'] / check['samples']
    widths = window_sizes(count, fs, 1000., 100, 10)
    last = np.zeros(widths.size, dtype=np.int64)
    counts = np.zeros(widths.size, dtype=np.int64)
    means, m2 = np.zeros(widths.size), np.zeros(widths.size)
    cumulative = squared = start = 0
    blocks = iter(reader.blocks(sid, chunk))
    current = next(blocks)
    while current is not None:
        following = next(blocks, None)
        prefix = np.cumsum(current, dtype=np.int64)
        _update_windows(prefix, start, cumulative, widths, last, counts, means, m2, center, 1., 0.)
        cumulative += int(prefix[-1])
        squared += int(np.square(current.astype(np.int64)).sum(dtype=np.int64))
        start += current.size
        current = following
        if (start // chunk) % 25 == 0:
            print(f'Source {sid}: analyzed {start:,}/{count:,}', flush=True)
    assert start == count
    assert np.all(counts == count // widths)
    check = metadata['source_checks'][sid]
    if (cumulative != check['integer_sum'] or squared != check['integer_sum_of_squares']
            or count != check['samples']):
        raise ValueError(f'Source {sid} samples changed between the measurement and analysis passes.')
    statistics = {int(w): dict(windows=int(counts[j]), mean=float(means[j]+center),
                    variance=float(m2[j]/(counts[j]-1)), m2=float(m2[j]),
                    used_samples=int(counts[j]*w), discarded_tail_samples=int(count-counts[j]*w))
                  for j, w in enumerate(widths)}
    return statistics


def analyze_waveforms(reader, chunk=4_000_000):
    metadata = reader.metadata
    counts = metadata['points_per_recording']
    if len(set(counts)) != 1:
        raise ValueError('This study requires four equal-length recordings.')
    # Start from samples on every run; ignore any stale in-memory statistics.
    metadata['source_checks'] = []
    metadata['centers_codes'] = []
    measured_insets, statistics = [], []
    for sid in range(4):
        check, trace = measure_source(reader, sid, chunk)
        metadata['source_checks'].append(check)
        metadata['centers_codes'].append(check['integer_sum'] / check['samples'])
        measured_insets.append(trace)
        statistics.append(analyze_source(reader, sid, chunk))
    reader.measured_insets = np.asarray(measured_insets)
    rows = []
    for sid in (1, 2, 3):
        for width, value in statistics[sid].items():
            base = statistics[0][width]
            rows.append(dict(source_index=sid, delta_c_ff=metadata['delta_c_ff'][sid],
                distance_codes=abs(value['mean'] - base['mean']),
                signed_center_codes=value['mean'], samples_per_window=width,
                tau_seconds=width/metadata['sample_rate_hz'], pair_samples=counts[sid],
                baseline_windows=base['windows'], signal_windows=value['windows'],
                mean_baseline=base['mean'], mean_signal=value['mean'],
                sigma_baseline=math.sqrt(base['variance']), sigma_signal=math.sqrt(value['variance']),
                baseline_used_samples=base['used_samples'], signal_used_samples=value['used_samples'],
                baseline_discarded_tail_samples=base['discarded_tail_samples'],
                signal_discarded_tail_samples=value['discarded_tail_samples']))
    return refresh_rows(rows, 'sample_count'), statistics


Original three-panel rendering, unchanged.


In [ ]:
def _short_number(value: float) -> str:
    for scale, suffix in ((1e9, "G"), (1e6, "M"), (1e3, "k")):
        if abs(value) >= scale:
            return f"{value/scale:g}{suffix}"
    return f"{value:g}"


def phosphor_opacity(hist: np.ndarray,
                     gamma: float = DEFAULT_BRIGHTNESS_GAMMA,
                     exposure: float = DEFAULT_BRIGHTNESS_EXPOSURE) -> np.ndarray:
    """Map line-hit density to color opacity; the default is linear (gamma=1).

    Let ref = max(1, percentile(positive_counts, 99.9)) and
    x = clip(counts / ref, 0, 1). Return clip(exposure * x**gamma, 0, 1).
    With gamma=1 and exposure=1, 10% density gives 10% opacity and
    50% density gives 50% opacity. Empty pixels remain white after compositing.

    This replaces the old 1-exp(-exposure*x**gamma) transfer entirely.
    The normalization percentile is unchanged; counts above ref saturate.
    Computation uses only the small density array and does not modify it.
    No graphics backend is imported and no figure or image is created here.
    """
    if not (math.isfinite(gamma) and gamma > 0
            and math.isfinite(exposure) and exposure > 0):
        raise ValueError("Gamma and exposure must be finite and positive.")
    values = np.asarray(hist)
    if values.dtype.kind not in "uif":
        raise ValueError("Histogram counts must be real numeric values.")
    if np.any(values < 0) or not np.all(np.isfinite(values)):
        raise ValueError("Histogram counts must be finite and nonnegative.")
    positive = values[values > 0]
    ref = (max(1.0, float(np.percentile(positive, BRIGHTNESS_REFERENCE_PERCENTILE)))
           if positive.size else 1.0)
    # float64 avoids needless precision loss in the uint64 normalization.
    # Work on a copy so saved density counts remain reusable and unchanged.
    alpha = values.astype(np.float64, copy=True)
    alpha /= ref
    np.clip(alpha, 0.0, 1.0, out=alpha)
    np.power(alpha, gamma, out=alpha)
    alpha *= exposure
    np.clip(alpha, 0.0, 1.0, out=alpha)
    return alpha


def phosphor_rgb(hist: np.ndarray, color: str,
                 gamma: float = DEFAULT_BRIGHTNESS_GAMMA,
                 exposure: float = DEFAULT_BRIGHTNESS_EXPOSURE) -> np.ndarray:
    """Single-color power-law opacity on white; ONE opaque PDF image XObject."""
    from matplotlib.colors import to_rgb
    alpha = phosphor_opacity(hist, gamma=gamma, exposure=exposure)
    rgb = 1.0 - alpha[..., None] * (1.0 - np.asarray(to_rgb(color), dtype=np.float64))
    return np.rint(np.clip(rgb, 0, 1) * 255).astype(np.uint8)


def font_settings(font_files: list[str] | None = None) -> tuple[dict, dict]:
    """Use PDF core Helvetica, or embed user-owned Helvetica TTF/OTF files."""
    settings = {"font.family": "sans-serif", "font.sans-serif": ["Helvetica"],
                "pdf.use14corefonts": True, "pdf.fonttype": 42, "ps.fonttype": 42,
                "text.usetex": False, "text.parse_math": False,
                "axes.unicode_minus": False, "path.simplify": False,
                "font.size": MIN_FONT_SIZE, "axes.labelsize": MIN_FONT_SIZE,
                "axes.titlesize": MIN_FONT_SIZE, "figure.titlesize": MIN_FONT_SIZE,
                "xtick.labelsize": MIN_FONT_SIZE, "ytick.labelsize": MIN_FONT_SIZE,
                "legend.fontsize": MIN_FONT_SIZE, "legend.title_fontsize": MIN_FONT_SIZE}
    info = {"family": "Helvetica", "mode": "PDF Base-14 Helvetica (not embedded)",
            "text_as_paths": False, "bitmap": "overview only", "inset": "vector zero-order hold",
            "minimum_font_size_pt": MIN_FONT_SIZE, "main_tick_font_size_pt": MIN_FONT_SIZE,
            "inset_tick_font_size_pt": MIN_FONT_SIZE, "inset_xlabel_font_size_pt": MIN_FONT_SIZE}
    if font_files:
        from matplotlib import font_manager
        families = []
        for filename in font_files:
            prop = font_manager.FontProperties(fname=filename)
            family = prop.get_name()
            if "helvetica" not in family.lower():
                raise ValueError(f"{filename}: font is {family!r}, not Helvetica.")
            font_manager.fontManager.addfont(filename)
            families.append(family)
        settings.update({"font.sans-serif": list(dict.fromkeys(families)), "pdf.use14corefonts": False})
        info["mode"] = "User-supplied Helvetica, subset-embedded (pdf.fonttype=42)"
    return settings, info


def _hour_axis_ticks(start: float, stop: float) -> tuple[np.ndarray, list[str]]:
    """Readable hour ticks including the true endpoints, without a crowded last tick."""
    from matplotlib.ticker import MaxNLocator
    if not (math.isfinite(start) and math.isfinite(stop) and start < stop):
        raise ValueError("Hour-axis bounds must be finite and increasing.")
    locator = MaxNLocator(nbins=6, min_n_ticks=3, steps=[1, 2, 2.5, 5, 10])
    candidates = locator.tick_values(start, stop)
    interior = candidates[(candidates > start) & (candidates < stop)]
    spacing = float(np.median(np.diff(candidates)))
    # Keep endpoint labels, omitting a nearby interior tick instead of overlapping.
    if interior.size and interior[0] - start < 0.55 * spacing:
        interior = interior[1:]
    if interior.size and stop - interior[-1] < 0.55 * spacing:
        interior = interior[:-1]
    ticks = np.concatenate(([start], interior, [stop]))
    minimum_gap = float(np.min(np.diff(ticks)))
    precision = max(2, 1 - math.floor(math.log10(minimum_gap)))
    labels = [f"{value:.{precision}f}".rstrip("0").rstrip(".") for value in ticks]
    return ticks, labels


def preview_pdf(pdf: str | Path, png: str | Path, dpi: int = 160) -> None:
    """Optional preview uses the actual PDF so Helvetica core-font metrics agree."""
    try:
        import fitz
    except ImportError as exc:
        raise RuntimeError("--preview requires PyMuPDF: python -m pip install pymupdf") from exc
    path = Path(png)
    if path.suffix.lower() != ".png":
        raise ValueError("Preview filename must end in .png.")
    path.parent.mkdir(parents=True, exist_ok=True)
    with fitz.open(pdf) as doc:
        doc[0].get_pixmap(dpi=dpi).save(path)
def render_study(hist, traces, metadata, rows, output, *, color='#2400ff', gamma=1., exposure=1., font_files=None):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.colors import to_rgb
    from matplotlib.lines import Line2D
    from matplotlib.patches import Rectangle, ConnectionPatch
    from matplotlib.text import Text
    from matplotlib.ticker import MaxNLocator, NullLocator
    output = Path(output)
    if output.suffix.lower() != '.pdf':
        raise ValueError('The study output must be PDF.')
    main, inset_view = View(**metadata['main_view']), Inset(**metadata['inset_view'])
    fs = metadata['sample_rate_hz']
    # Recompute model probabilities from the current paired-window statistics
    # on every render, including archive replots. Stored curve values are never
    # used as a historical reference or retained across a change in d-prime.
    support_mode = metadata.get('analysis', {}).get('support_mode', 'sample_count')
    rows = refresh_rows(rows, support_mode=support_mode)
    support_label = '1/N' if support_mode == 'sample_count' else '1/window count'
    waveform_threshold = metadata.get('waveform_threshold_codes', 5.)
    curve_segments = {}
    # One source/color table drives the density, inset, and analysis panels.
    # A legacy 2-D archive contains merged counts and cannot be recolored by
    # source; preserve its original single-color rendering when replotting it.
    source_colors = ('#000000', '#0072BD', '#D95319', '#009E73')
    density = np.asarray(hist)
    separate_density = density.ndim == 3
    visible_sources = tuple(metadata.get('main_panel_sources',
                            range(4) if separate_density else VISIBLE_SOURCES))
    if not visible_sources or any(sid not in range(4) for sid in visible_sources):
        raise ValueError('Visible source indices must be drawn from the four study recordings.')
    if separate_density:
        if density.shape != (4, main.ny, main.nx):
            raise ValueError('Per-source density must have shape (4, main.ny, main.nx).')
        rgb = np.ones((main.ny, main.nx, 3), dtype=np.float64)
        for sid in visible_sources:
            opacity = phosphor_opacity(density[sid], gamma=gamma, exposure=exposure)[..., None]
            rgb = rgb * (1.0 - opacity) + np.asarray(to_rgb(source_colors[sid])) * opacity
        waveform_rgb = np.rint(np.clip(rgb, 0.0, 1.0) * 255).astype(np.uint8)
    else:
        if density.shape != (main.ny, main.nx):
            raise ValueError('Legacy density must have shape (main.ny, main.nx).')
        waveform_rgb = phosphor_rgb(density, color, gamma=gamma, exposure=exposure)

    def plot_model_curve(axis, tau, values, pfa, pfa_limit, clr, marker):
        """Apply the notebook's segment support rule without changing styling."""
        tiny = np.finfo(float).tiny
        supported = ((np.sqrt(tau[:-1] * tau[1:]) >= 1. / fs)
                     & ((np.log(np.clip(pfa[:-1], tiny, None))
                         + np.log(np.clip(pfa[1:], tiny, None)))
                        >= (np.log(np.clip(pfa_limit[:-1], tiny, None))
                            + np.log(np.clip(pfa_limit[1:], tiny, None)))))
        if supported.size:
            starts = np.r_[0, np.flatnonzero(supported[1:] != supported[:-1]) + 1]
            ends = np.r_[starts[1:], supported.size]
            for start, end in zip(starts, ends):
                within_support = bool(supported[start])
                axis.plot(tau[start:end+1], values[start:end+1],
                          '-' if within_support else '--', color=clr,
                          linewidth=2.3 if within_support else 1.9, zorder=3)
        # Keep the previous visual density despite the denser calculation grid;
        # symbols identify a trace independently of its solid/dashed sections.
        marker_indices = np.unique(np.linspace(0, len(tau)-1, min(11, len(tau)), dtype=int))
        axis.plot(tau[marker_indices], values[marker_indices], linestyle='none',
                  marker=marker, color=clr, markersize=6, markerfacecolor='white', zorder=4)
        return dict(solid_segments=int(supported.sum()),
                    dashed_segments=int((~supported).sum()))

    rc, font_info = font_settings(font_files)
    with plt.rc_context(rc):
        fig = plt.figure(figsize=(22.8, 7.0), facecolor='white')
        try:
            ax = fig.add_axes([.061, .185, .350, .735])
            sep = fig.add_axes([.522, .185, .185, .735])
            err = fig.add_axes([.797, .185, .192, .735])
            extent = (main.start/fs/3600, main.stop/fs/3600, main.ymin, main.ymax)
            ax.imshow(waveform_rgb,
                      extent=extent, origin='lower', aspect='auto', interpolation='none')
            ax.set_xlim(extent[:2]); ax.set_ylim(extent[2:])
            ticks, labels = _hour_axis_ticks(*extent[:2])
            ax.set_xticks(ticks, labels)
            ax.set_yticks(metadata.get('main_y_ticks_codes', [-30, -20, -10, 0, 10, 20]))
            ax.set_xlabel('Time / hours', fontsize=26, fontweight='bold', fontstyle='italic', labelpad=13)
            ax.set_ylabel(f'Output code at {_short_number(fs)}S/s', fontsize=25, fontweight='bold', fontstyle='italic', labelpad=13)
            # Threshold is a vector guide, not a third waveform.
            ax.axhline(waveform_threshold, color='black', linestyle='--', linewidth=1.2, alpha=.65)
            pos = ax.get_position()
            ins = fig.add_axes([pos.x0+.285*pos.width, pos.y0+.13*pos.height,
                                .665*pos.width, .27*pos.height], zorder=5, facecolor='white')
            plt.sca(ins)
            for sid in visible_sources:
                # Sample n is held on [n, n+1); the last sample is held through
                # the exclusive right boundary instead of ending at count-1.
                trace = np.asarray(traces[sid])
                if trace.shape != (inset_view.count,):
                    raise ValueError('Inset trace length does not match its sample window.')
                ins.step(np.arange(inset_view.count + 1), np.r_[trace, trace[-1]],
                         where='post', color=source_colors[sid] if separate_density else color, linewidth=.8,
                         solid_capstyle='butt', rasterized=False)
            ins.set_xlim(0, inset_view.count)
            ins.set_ylim(inset_view.ymin, inset_view.ymax)
            offsets = [0, 50, 100, 150, 200] if inset_view.count==200 else np.unique(np.rint(np.linspace(0, inset_view.count, 5)).astype(int))
            ins.set_xticks(offsets, [f'{inset_view.start/fs/3600:g} h']+[f'+{i}' for i in offsets[1:]])
            ins.set_xlabel('Sample Offset', fontsize=18, labelpad=5)
            ins.yaxis.set_major_locator(MaxNLocator(nbins=3, integer=True))
            z0, z1 = inset_view.start/fs/3600, inset_view.stop/fs/3600
            ax.add_patch(Rectangle((z0, inset_view.ymin), z1-z0, inset_view.ymax-inset_view.ymin,
                                   fill=False, edgecolor='black', linewidth=1.7, zorder=4))
            for x, corner in ((z0, (0, 1)), (z1, (1, 1))):
                ax.add_artist(ConnectionPatch(xyA=(x, inset_view.ymin), coordsA='data', axesA=ax,
                               xyB=corner, coordsB='axes fraction', axesB=ins, color='black', linewidth=1.7, zorder=4))
            for text in ins.get_yticklabels():
                text.set_bbox(dict(facecolor='white', edgecolor='none', pad=.5))
            # Reference-style color/marker encoding, identical on both analysis panels.
            colors = source_colors[1:]
            markers = ('o', 's', '^')
            for sid, clr, marker in zip((1, 2, 3), colors, markers):
                curve = sorted((r for r in rows if r['source_index']==sid),
                               key=lambda r: r['tau_seconds'])
                tau = np.array([r['tau_seconds'] for r in curve])
                ds = np.array([r['dprime'] for r in curve])
                pfa = np.array([r['gaussian_false_positive_rate'] for r in curve])
                pfa_limit = np.array([r['pfa_limit'] for r in curve])
                curve_segments[str(sid)] = plot_model_curve(
                    sep, tau, ds, pfa, pfa_limit, clr, marker)
                plot_model_curve(err, tau, np.maximum(pfa, np.finfo(float).tiny),
                                 pfa, pfa_limit, clr, marker)
            sep.set_ylabel("Detection separability, d'", fontsize=24, fontweight='bold', fontstyle='italic', labelpad=12)
            err.set_ylabel('False positive probability', fontsize=24, fontweight='bold', fontstyle='italic', labelpad=11)
            sep.set_ylim(0., 1.0*max(r['dprime'] for r in rows))
            sep.yaxis.set_major_locator(MaxNLocator(nbins=6))
            err.set_yscale('log'); err.set_ylim(1e-12, 1e-1)
            yt = [1., 1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12]
            err.set_yticks(yt, ['1', '1e-2', '1e-4', '1e-6', '1e-8', '1e-10', '1e-12'])
            err.yaxis.set_minor_locator(NullLocator())
            err.axhline(1e-8, linestyle=':', color='black', linewidth=1.6)
            for axis in (sep, err):
                axis.set_xscale('log')
                axis.set_xlim(1e-4, 100.)
                choices = [(1e-4, '100 \u00b5'), (1e-3, '1 m'), (1e-2, '10 m'), (1., '1'), (100., '100')]
                choices = [(v, lab) for v, lab in choices if axis.get_xlim()[0] <= v <= axis.get_xlim()[1]]
                if len(choices)>=2:
                    axis.set_xticks([v for v, _ in choices], [lab for _, lab in choices])
                axis.xaxis.set_minor_locator(NullLocator())
                axis.set_xlabel('Integration time / s', fontsize=24, fontweight='bold', fontstyle='italic', labelpad=12)
                axis.grid(True, which='major', linestyle=':', linewidth=.8, alpha=.7)
            legend_handles = [Line2D([], [], color=clr, marker=marker, markerfacecolor='white',
                                    linewidth=2.3, markersize=6, label=f"Delta C = {next(r['delta_c_ff'] for r in rows if r['source_index']==sid):g} fF")
                              for sid, clr, marker in zip((1, 2, 3), colors, markers)]
            legend_handles.append(Line2D([], [], color='black', linewidth=1.9,
                                         linestyle='--', label='Gaussian'))
            for axis, location in ((sep, 'lower right'), (err, 'upper right')):
                axis.legend(handles=legend_handles, loc=location, fontsize=18, framealpha=.94,
                            handlelength=2.1, borderpad=.4, labelspacing=.3)
            fig.text(.522, .024, f'Gaussian estimates; dashed below {support_label}',
                     fontsize=18, ha='left', va='bottom')
            err.set_title('Midpoint thresholds; Gaussian model', fontsize=18, pad=11)
            for axis in (ax, sep, err, ins):
                axis.tick_params(labelsize=18, length=6, width=1.3, pad=5)
                for spine in axis.spines.values():
                    spine.set_linewidth(1.35)
            for text in fig.findobj(match=Text):
                if text.get_fontsize()<18:
                    text.set_fontsize(18)
            output.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(output, format='pdf', dpi=200, facecolor='white',
                        metadata=dict(Title='Connected-line DPO and midpoint-threshold capacitance detection',
                                      Creator=f'phosphor.py {VERSION} / capacitance_study.py / Matplotlib',
                                      Subject='Bitmap waveform overview; vector axes, zero-order-hold inset, separability and Gaussian false-positive curves'))
        finally:
            plt.close(fig)
    return dict(font_info, main_panel_sources=list(visible_sources), bitmap_count=1,
                main_ylim_codes=[float(v) for v in ax.get_ylim()],
                main_y_ticks_codes=[float(v) for v in ax.get_yticks()],
                waveform_source_colors={str(sid): source_colors[sid] if separate_density else color
                                        for sid in visible_sources},
                density_compositing='per-source opacity over white in source order' if separate_density else 'legacy merged single color',
                native_bitmap=[main.nx, main.ny], all_analysis_curves='vector',
                gamma=gamma, exposure=exposure, waveform_threshold_codes=waveform_threshold,
                decision_threshold='midpoint of each paired recording mean at each integration time',
                inset_rendering='vector zero-order hold; each sample held for one full sample interval',
                inset_hold_intervals=inset_view.count, identical_analysis_legends=True,
                dynamic_gaussian=True, gaussian_curves_source='current paired-window statistics',
                gaussian_curve_points=len(rows), gaussian_probability='ndtr(direction*(mean_baseline-threshold)/sigma_baseline)',
                curve_classification=f'on-the-fly geometric segment midpoint comparison with {support_label}',
                support_mode=support_mode, curve_segments=curve_segments,
                separability_xlim_seconds=[float(v) for v in sep.get_xlim()],
                separability_major_ticks_seconds=[float(v) for v in sep.get_xticks()],
                inset_xlim_samples=[float(v) for v in ins.get_xlim()],
                inset_major_ticks_samples=[int(v) for v in ins.get_xticks()],
                probability_xlim_seconds=[float(v) for v in err.get_xlim()],
                probability_major_ticks_seconds=[float(v) for v in err.get_xticks()],
                marker_count_per_curve=11)


In [ ]:
reader = WaveformInput(INPUT, verify_checksums=VERIFY_CHECKSUMS)
with np.load(OUTPUT_DIR / 'study_density.npz', allow_pickle=False) as saved:
    hist, traces = saved['hist'], saved['traces']
main, inset = View(**SETTINGS['main_view']), Inset(**SETTINGS['inset_view'])
main.validate()
inset.validate()
if (hist.shape != (4, main.ny, main.nx) or hist.dtype.kind not in 'iu'
        or np.any(hist < 0) or traces.shape != (4, inset.count)
        or not np.isfinite(traces).all()):
    raise ValueError('Cached density or inset does not match the plotting settings; rerun 01_plot.')
rows, statistics = analyze_waveforms(reader, ANALYSIS_CHUNK)
np.testing.assert_array_equal(traces, reader.measured_insets,
    err_msg='Cached inset differs from waveform samples; rerun 01_plot.')
metadata = json.loads(json.dumps(reader.metadata))
metadata['waveform_input_sha256'] = reader.fingerprint
metadata['density_sha256'] = sha256_file(OUTPUT_DIR / 'study_density.npz')
metadata['execution'] = dict(sample_decimation=False,
    samples_analyzed=sum(check['samples'] for check in metadata['source_checks']),
    analysis_chunk=ANALYSIS_CHUNK, all_window_statistics_recomputed=True,
    full_record_moments_measured_from_samples=True, inset_verified_against_samples=True)
pdf = OUTPUT_DIR / 'capacitance_detection_noise_1p5_irregular_drift.pdf'
metadata['render'] = render_study(hist, traces, metadata, rows, pdf, **metadata['render_settings'])
preview_pdf(pdf, pdf.with_suffix('.png'), dpi=130)
(OUTPUT_DIR / 'detection_rows.json').write_text(json.dumps(rows, indent=2)+'\n')
with (OUTPUT_DIR / 'detection_metrics.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
for sid, stats in enumerate(statistics):
    source = dict(source_index=sid, independent_processed_moments=metadata['source_checks'][sid], window_statistics=stats,
        density_sha256=sha256_file(OUTPUT_DIR / f'source_{sid}_density.npz'),
        inset_matches_processed_samples=True)
    (OUTPUT_DIR / f'source_{sid}.json').write_text(json.dumps(source, indent=2)+'\n')
import sys
import scipy
import numba
import matplotlib
import fitz
environment = dict(python=sys.version, packages=dict(numpy=np.__version__,
    scipy=scipy.__version__, numba=numba.__version__, matplotlib=matplotlib.__version__,
    pymupdf=fitz.VersionBind), local_timezone=list(time.tzname),
    completed_at=time.strftime('%Y-%m-%dT%H:%M:%S%z'))
(OUTPUT_DIR / 'execution_environment.json').write_text(json.dumps(environment, indent=2)+'\n')
(OUTPUT_DIR / 'study.json').write_text(json.dumps(metadata, indent=2)+'\n')
print(f'Saved full figure and all analysis results to {OUTPUT_DIR.resolve()}')

try:
    from IPython.display import Image, display
except ImportError:
    print(f'Preview: {pdf.with_suffix(".png").resolve()}')
else:
    display(Image(filename=str(pdf.with_suffix('.png'))))
